# W4A16 Quantization Lab — Qwen3-4B-Instruct-2507

Quantize model weights from BF16 → INT4 (per-group, group_size=128, symmetric).

**Lab steps**
1. Load BF16 baseline → measure memory and PPL
2. Implement per-group symmetric quantizer
3. Sketch a naive `W4A16Linear` (stores INT4 as int8 — simple but wastes memory)
4. Build production `WQLinear` (packs 2 INT4 values per uint8 byte)
5. Save quantized model to disk and reload without the FP weights
6. Compare PPL and generation quality: FP vs W4A16

## 1  Imports & helpers

In [ ]:
import math
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from accelerate import init_empty_weights, load_checkpoint_and_dispatch

In [ ]:
def compute_ppl(
    model,
    tokenizer,
    dataset_name: str = "wikitext",
    dataset_config: str = "wikitext-2-raw-v1",
    split: str = "test",
    max_length: int = 2048,
    stride: int = 512,
    device=None,
    verbose: bool = True,
) -> float:
    """Perplexity on wikitext-2 via sliding window.


    Overlapping windows (stride < max_length) score each token with maximum
    left-context, avoiding the inflated NLL from cold-start prefix tokens.
    """
    if device is None:
        device = next(model.parameters()).device


    data = load_dataset(dataset_name, dataset_config, split=split)
    text = "\n\n".join(data["text"])
    input_ids = tokenizer(text, return_tensors="pt").input_ids  # (1, N)
    total_tokens = input_ids.size(1)


    if verbose:
        print(f"Corpus: {total_tokens:,} tokens | window={max_length} stride={stride}")


    nlls, prev_end, window_idx = [], 0, 0


    for begin in range(0, total_tokens, stride):
        end = min(begin + max_length, total_tokens)
        chunk = input_ids[:, begin:end].to(device)
        target_len = end - prev_end  # new tokens only — no double-counting overlap


        with torch.no_grad():
            logits = model(chunk).logits  # (1, seq, vocab)


        # logits[t] predicts token[t+1] → shift by 1
        shift_logits = logits[:, -target_len:-1, :].contiguous()
        shift_labels = chunk[:, -target_len + 1:].contiguous()


        nll = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            reduction="sum",
        )
        nlls.append(nll.float())  # fp32 to avoid overflow on large window sums


        if verbose and window_idx % 10 == 0:
            running = torch.exp(torch.stack(nlls).sum() / max(1, end - 1))
            print(f"  window {window_idx:4d} | tokens {end:>8,}/{total_tokens:,} | PPL {running:.2f}")


        prev_end = end
        window_idx += 1
        if end == total_tokens:
            break


    ppl = torch.exp(torch.stack(nlls).sum() / (total_tokens - 1))
    return ppl.item()

In [ ]:
def model_size_gb(model) -> float:
    """Parameters + buffers. Buffers hold qweight/qzeros/scales in quantized layers."""
    total = sum(p.numel() * p.element_size() for p in model.parameters())
    total += sum(b.numel() * b.element_size() for b in model.buffers())
    return total / 1e9

## 2  FP baseline — load model, measure memory and PPL

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_fp = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="cuda",
)
model_fp.eval()
print(f"FP model: {model_size_gb(model_fp):.2f} GB")

In [ ]:
ppl_fp = compute_ppl(model_fp, tokenizer)
print(f"\nFP baseline PPL: {ppl_fp:.2f}")

## 3  Per-group symmetric quantization

Map each group of `group_size` weights to the INT4 range `[-8, 7]`:

```
scale = max(|W_group|) / 7
q     = clamp(round(W_group / scale), -8, 7)
```

Group size 128 means one scale per 128 weights — fine-grained enough to handle
per-channel magnitude variation without per-element overhead.
Symmetric means zero-point = 0 (no asymmetric offset needed).

In [ ]:
def quantize_w4_groupwise_sym(W: torch.Tensor, group_size: int = 128):
    """
    Symmetric per-group INT4 quantization.


    Args:
        W:          weight tensor, shape (out_features, in_features)
        group_size: number of weights sharing one scale


    Returns:
        q          – INT4 values stored as int8, shape (num_groups, group_size)
        scales     – fp16 per-group scales,     shape (num_groups, 1)
        orig_shape – W.shape  (needed to reshape after dequant)
        orig_numel – number of real elements before padding
    """
    W = W.float()
    orig_shape = W.shape
    flat = W.flatten()
    orig_numel = flat.numel()


    num_groups = math.ceil(orig_numel / group_size)
    padded = num_groups * group_size
    if padded > orig_numel:
        flat = torch.cat([flat, flat.new_zeros(padded - orig_numel)])


    grouped = flat.view(num_groups, group_size)
    scales = (grouped.abs().amax(dim=1, keepdim=True) / 7.0).clamp(min=1e-8)
    q = torch.clamp(torch.round(grouped / scales), -8, 7).to(torch.int8)


    return q, scales.half(), orig_shape, orig_numel

## 4  Naive W4A16Linear (pedagogical sketch)

The simplest quantized layer: quantize on construction, store INT4 values in an
**int8** buffer (one byte per value — correct but wastes the upper nibble), dequantize
on every forward pass.

**Limitation for checkpointing:** `__init__` requires a live `nn.Linear` to quantize
from. Reloading means loading the full FP model first — defeating the purpose of
a small checkpoint.

The production `WQLinear` in §5 solves this with `from_linear(..., init_only=True)`.

In [ ]:
class W4A16Linear(nn.Module):
    """Naive INT4 layer: one int8 byte per INT4 value (unpacked, simple but wasteful)."""


    def __init__(self, linear: nn.Linear, group_size: int = 128):
        super().__init__()
        self.group_size = group_size


        W = linear.weight.data.float()
        self.out_shape = W.shape


        q, scales, _, orig_numel = quantize_w4_groupwise_sym(W, group_size)
        self.orig_numel = orig_numel


        self.register_buffer("w_int8", q)      # int8 storing INT4 values
        self.register_buffer("scale", scales)  # fp16, shape (num_groups, 1)
        self.bias = linear.bias


    def forward(self, x):
        # dequantize: multiply, slice away any padding, reshape to original weight shape
        w = (self.w_int8.float() * self.scale.float())
        w = w.flatten()[:self.orig_numel].reshape(self.out_shape)
        return F.linear(x, w.to(x.dtype), self.bias)

## 5  Production WQLinear — packed INT4 (2 values per byte)

INT4 has no native dtype. Storing one INT4 in an int8 wastes the upper nibble.
The fix: pack two INT4 values into one uint8.

```
byte  =  (lo & 0xF)  |  ((hi & 0xF) << 4)
```

| nibble | bits | holds |
|--------|------|-------|
| low    | 0–3  | even-indexed element |
| high   | 4–7  | odd-indexed element  |

**`& 0xF` mask:** clamps any overflow to 4 bits before packing, and strips garbage
bits above bit 3 after unpacking.

**Sign-extension on unpack:** INT4 uses two's complement, so nibble values 8–15
represent −8 to −1. After unpacking to int32, subtract 16 from any value > 7.

```
Pack  3 (lo) and 9 (hi):
  3        = 0000 0011
  9 << 4   = 1001 0000
  OR       = 1001 0011  = 0x93

Unpack 0x93:
  lo: 0x93 & 0x0F = 0000 0011 = 3  ✓
  hi: 0x93 >> 4   = 0000 1001 = 9  ✓
```

In [ ]:
class WQLinear(nn.Module):
    def __init__(self, w_bit, group_size, in_features, out_features, bias):
        super().__init__()
        self.w_bit = w_bit
        self.group_size = group_size
        self.in_features = in_features
        self.out_features = out_features


        # 2 INT4 values packed per uint8 → out_features // 2 columns
        self.register_buffer("qweight", torch.zeros(in_features, out_features // 2, dtype=torch.uint8))
        self.register_buffer("qzeros",  torch.zeros(in_features // group_size, out_features // 2, dtype=torch.uint8))
        self.register_buffer("scales",  torch.zeros(in_features // group_size, out_features, dtype=torch.float16))
        if bias:
            self.register_buffer("bias", torch.zeros(out_features, dtype=torch.float16))
        else:
            self.bias = None


    @staticmethod
    def _pack(x_int):
        """(rows, cols) int32 → (rows, cols//2) uint8 — 2 nibbles per byte."""
        return ((x_int[:, 0::2] & 0xF) | ((x_int[:, 1::2] & 0xF) << 4)).to(torch.uint8)


    @staticmethod
    def _unpack(packed):
        """(rows, cols//2) uint8 → (rows, cols) int32, sign-extended."""
        out = torch.zeros(packed.shape[0], packed.shape[1] * 2,
                          dtype=torch.int32, device=packed.device)
        out[:, 0::2] = packed.int() & 0xF
        out[:, 1::2] = (packed.int() >> 4) & 0xF
        out[out > 7] -= 16  # sign-extend: nibbles 8–15 → −8 to −1
        return out


    @classmethod
    def from_linear(cls, linear, w_bit, group_size, scales, zeros, init_only=False):
        """
        linear    : nn.Linear being replaced
        scales    : (in_features // group_size, out_features)  float16
        zeros     : (in_features // group_size, out_features)  float16
        init_only : if True, allocate empty buffers only (load-from-disk path)
        """
        q = cls(w_bit, group_size, linear.in_features, linear.out_features,
                bias=linear.bias is not None)


        if init_only:
            return q  # buffers filled later by load_checkpoint_and_dispatch


        q.scales = scales.clone().half()


        W = linear.weight.data.float().t()  # (in_features, out_features)
        W_int = torch.zeros(linear.in_features, linear.out_features, dtype=torch.int8)
        Z_int = zeros.to(torch.int8)


        for g in range(linear.in_features // group_size):
            s  = scales[g]                                          # (out,)
            z  = zeros[g]                                           # (out,)
            wg = W[g * group_size : (g + 1) * group_size]          # (group_size, out)
            W_int[g * group_size : (g + 1) * group_size] = \
                torch.clamp(torch.round(wg / s + z), -8, 7).to(torch.int8)


        q.qweight = cls._pack(W_int)
        q.qzeros  = cls._pack(Z_int)


        if linear.bias is not None:
            q.bias = linear.bias.clone().half()


        return q


    def forward(self, x):
        W_int = self._unpack(self.qweight)  # (in_features, out_features)  int32
        Z_int = self._unpack(self.qzeros)   # (in//g, out_features)        int32


        # broadcast scales/zeros from (in//g, out) → (in, out)
        s = self.scales.repeat_interleave(self.group_size, dim=0)       # fp16
        z = Z_int.half().repeat_interleave(self.group_size, dim=0)      # fp16


        W_fp = s * (W_int.half() - z)  # dequantize → (in, out) fp16


        out = x.half() @ W_fp
        if self.bias is not None:
            out = out + self.bias
        return out.to(x.dtype)

## 6  Quantize the FP model

Two helpers:
- `build_scales_zeros` — compute per-group scales from the FP weights (zeros = 0 for symmetric)
- `replace_linear_with_wqlinear` — walk the module tree and swap every `nn.Linear` for a `WQLinear`

In [ ]:
def get_parent(model, name):
    """Return the parent module of a dotted path like 'model.layers.0.mlp.gate_proj'."""
    parent = model
    for part in name.split(".")[:-1]:
        parent = getattr(parent, part)
    return parent


def build_scales_zeros(model, group_size=128):
    scales_dict, zeros_dict = {}, {}
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        W = module.weight.data.float().t()  # (in, out)
        in_features, out_features = W.shape
        n_groups = in_features // group_size
        W_grouped = W.view(n_groups, group_size, out_features)
        scales = (W_grouped.abs().amax(dim=1) / 7.0).clamp(min=1e-8)  # (n_groups, out)
        scales_dict[name] = scales.half()
        zeros_dict[name]  = torch.zeros_like(scales).half()  # symmetric → zero-point = 0
    return scales_dict, zeros_dict


def replace_linear_with_wqlinear(model, w_bit, group_size, scales_dict, zeros_dict):
    model = copy.deepcopy(model)  # don't mutate model_fp — we still need it for PPL comparison
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear):
            parent = get_parent(model, name)
            attr = name.split(".")[-1]
            wq = WQLinear.from_linear(
                module,
                w_bit=w_bit,
                group_size=group_size,
                scales=scales_dict[name],
                zeros=zeros_dict[name],
            )
            setattr(parent, attr, wq)
    return model

In [ ]:
scales_dict, zeros_dict = build_scales_zeros(model_fp)
model_q = replace_linear_with_wqlinear(model_fp, w_bit=4, group_size=128,
                                        scales_dict=scales_dict, zeros_dict=zeros_dict)
model_q.eval()
print(f"FP model:    {model_size_gb(model_fp):.2f} GB")
print(f"W4A16 model: {model_size_gb(model_q):.2f} GB")

## 7  Save to disk

In [ ]:
SAVE_DIR = "./qwen3-4b-w4a16"


model_q.config.quantization_config = {
    "quant_method": "wqlinear",
    "w_bit": 4,
    "group_size": 128,
}
model_q.save_pretrained(SAVE_DIR)   # writes config.json + model.safetensors
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")

## 8  Load from disk (no FP model needed)

Four-step recipe that avoids loading FP weights:
1. Read `quantization_config` from `config.json`
2. Build an empty meta-device model skeleton
3. Replace every `nn.Linear` with a shape-only `WQLinear` (`init_only=True`)
4. Fill all buffers from the safetensors checkpoint

In [ ]:
SAVE_DIR = "./qwen3-4b-w4a16"


# Step 1: read quant config
config = AutoConfig.from_pretrained(SAVE_DIR)
quant_cfg = config.quantization_config


# Step 2: empty skeleton (no weights allocated)
with init_empty_weights():
    model_loaded = AutoModelForCausalLM.from_config(config)


# Step 3: replace nn.Linear with shape-only WQLinear
for name, module in list(model_loaded.named_modules()):
    if isinstance(module, nn.Linear):
        parent = get_parent(model_loaded, name)
        attr = name.split(".")[-1]
        wq = WQLinear.from_linear(
            module,
            w_bit=quant_cfg["w_bit"],
            group_size=quant_cfg["group_size"],
            scales=None,
            zeros=None,
            init_only=True,
        )
        setattr(parent, attr, wq)


# Step 4: fill buffers from checkpoint
load_checkpoint_and_dispatch(model_loaded, checkpoint=SAVE_DIR, device_map="auto")
model_loaded.eval()
print(f"Loaded W4A16: {model_size_gb(model_loaded):.2f} GB")

## 9  Evaluate — FP vs W4A16

In [ ]:
ppl_w4 = compute_ppl(model_loaded, tokenizer)


print(f"\n{'Model':<12}  {'PPL':>8}  {'Size':>9}")
print(f"{'FP (BF16)':<12}  {ppl_fp:>8.2f}  {model_size_gb(model_fp):>7.2f} GB")
print(f"{'W4A16':<12}  {ppl_w4:>8.2f}  {model_size_gb(model_loaded):>7.2f} GB")
print(f"{'PPL delta':<12}  {ppl_w4 - ppl_fp:>+8.2f}")

In [ ]:
def generate_response(model, prompt: str, max_new_tokens: int = 256) -> str:
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(next(model.parameters()).device)
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True)


PROMPT = "Explain weight quantization in one paragraph."


resp_fp = generate_response(model_fp, PROMPT)
resp_w4 = generate_response(model_loaded, PROMPT)


print("=== FP (BF16) ===")
print(resp_fp)
print("\n=== W4A16 ===")
print(resp_w4)